In [0]:
# ==========================================================
# PAYCE DATA PLATFORM
# NOTEBOOK: 01_bronze_autoloader
# DESCRIPTION: Ingesting raw data from S3 into Delta Bronze
# AUTHOR: Atharva Bhagwat Payce DE 
# ==========================================================


# AWS S3 path is defined
S3_RAW_BASE = "s3://payce-data-platform/raw/"
S3_BRONZE_BASE = "s3://payce-data-platform/bronze/"

# Verify Connection - list raw folder
dbutils.fs.ls("s3://payce-data-platform/raw/")

In [0]:
# BRONZE LAYER - PaySim Ingestion
# SOURCE: s3://payce-data-platform/raw/paysim/
# TARGET: s3://payce-data-platform/raw/paysim/

# Read raw PaySim data from s3 
df_paysim = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{S3_BRONZE_BASE}/paysim/_schema")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{S3_RAW_BASE}/paysim/")
)

# Write raw PaySim data into Delta Bronze Table
df_paysim_bronze = (
    df_paysim.writeStream
    .format("delta")
    .option("checkpointLocation", f"{S3_BRONZE_BASE}/paysim/_checkpoint")
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .start(f"{S3_BRONZE_BASE}/paysim")
)

In [0]:
# Verify PaySim Bronze Table
df_verify = spark.read.format("delta").load(f"{S3_BRONZE_BASE}/paysim/") 

print(f"Total rows ingested: {df_verify.count():,}")


In [0]:
# ===============================================================
# BRONZE LAYER - IEEE CIS Fraud Detection Ingestion
# Source: s3://payce-data-platform/raw/ieee_cis/
# Target: s3://payce-data-platform/bronze/ieee_cis/
# ===============================================================

# Read raw IEEE-CIS data using AutoLoader
df_ieee = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{S3_BRONZE_BASE}/ieee_cis/_schema")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{S3_RAW_BASE}/ieee_cis/")
)

# Write raw IEEE-CIS data into Delta Bronze Table
df_ieee_bronze = (
    df_ieee.writeStream
    .format("delta")
    .option("checkpointLocation", f"{S3_BRONZE_BASE}/ieee_cis/_checkpoint")
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .start(f"{S3_BRONZE_BASE}/ieee_cis/")
)

In [0]:
# Verify IEEE-CIS Bronze Table
df_verify_ieee = spark.read.format("delta").load(f"{S3_BRONZE_BASE}/ieee_cis/")

print(f"Total  rows ingested:{df_verify_ieee.count():,}")

In [0]:
print(f"Columns: {df_verify_ieee.columns}")

In [0]:
df_verify_ieee.show(5)

In [0]:
# ==============================================================
# BRONZE LAYER - Home Credit Default Risk Ingestion
# Source: s3://payce-data-platform/raw/home_credit/
# Target: s3://payce-data-platform/bronze/home_credit/
# ==============================================================

# Read raw data using AutoLoader
df_home_credit = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{S3_BRONZE_BASE}/home_credit/_schema")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{S3_RAW_BASE}/home_credit/")
)

# Write raw data into Delta Bronze Table
df_home_credit_bronze = (
    df_home_credit.writeStream
    .format("delta")
    .option("checkpointLocation", f"{S3_BRONZE_BASE}/home_credit/_schema")
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .start(f"{S3_BRONZE_BASE}/home_credit/")
)

In [0]:
# Verify Home Credit Bronze Table
df_verify_home_credit = spark.read.format("delta").load(f"{S3_BRONZE_BASE}/home_credit/")

print(f"Total rows ingested: {df_verify_home_credit.count():,}")

In [0]:
print(f"Columns: {df_verify_home_credit.columns}")

In [0]:
display(df_verify_home_credit.show(5))

In [0]:
# ============================================================
# BRONZE LAYER — LendingClub Loan Dataset Ingestion
# Source: s3://payce-data-platform/raw/lending_club/
# Target: s3://payce-data-platform/bronze/lending_club/
# ============================================================

import re

def clean_column_names(df):
    """Remove invalid characters from column names for Delta compatiblity"""
    cleaned_columns = [
        re.sub(r'[ ,;{}()\n\t=]+', '_', col).strip('_')
        for col in df.columns
    ]
    return df.toDF(*cleaned_columns)


# Read raw LendingClub data using Auto Loader
df_lending = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{S3_BRONZE_BASE}/lending_club/_schema")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{S3_RAW_BASE}/lending_club/")
)

# Clean column names
df_lending_cleaned = clean_column_names(df_lending)


# Write to Delta Bronze table
(
    df_lending_cleaned.writeStream
    .format("delta")
    .option("checkpointLocation", f"{S3_BRONZE_BASE}/lending_club/_checkpoint")
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .start(f"{S3_BRONZE_BASE}/lending_club/")
)

In [0]:
dbutils.fs.rm(f"{S3_BRONZE_BASE}/lending_club/", True)
print("✅ Cleared")

In [0]:
# ============================================================
# BRONZE LAYER — LendingClub Loan Dataset Ingestion
# Using batch read instead of streaming
# ============================================================

import re

# Read as batch
df_lending = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{S3_RAW_BASE}/lending_club/")
)

# Clean column names
new_cols = [re.sub(r'[^a-zA-Z0-9_]', '_', c).strip('_') for c in df_lending.columns]
df_lending = df_lending.toDF(*new_cols)

# Write to Delta Bronze
(
    df_lending.write
    .format("delta")
    .mode("overwrite")
    .save(f"{S3_BRONZE_BASE}/lending_club/")
)

print("✅ LendingClub ingested successfully")

In [0]:
# Verify LendingClub Bronze table
df_verify_lc = spark.read.format("delta").load(f"{S3_BRONZE_BASE}/lending_club/")

print(f"Total rows ingested: {df_verify_lc.count():,}")
df_verify_lc.show(5)